In [1]:
import numpy as np

class MySGDRegressor:
    def __init__(self, lr=0.01, max_iter=1000, shuffle=True):
        """
        lr: 学习率
        max_iter: 迭代整个数据集的次数（epoch 数）
        shuffle: 每轮是否打乱数据
        """
        self.lr = lr
        self.max_iter = max_iter
        self.shuffle = shuffle
        self.w = None
        self.b = None

    def _shuffle_data(self, X, y):
        """打乱数据"""
        idx = np.random.permutation(len(X))
        return X[idx], y[idx]

    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)

        n_samples, n_features = X.shape
        
        # 初始化参数
        self.w = np.zeros(n_features)
        self.b = 0.0

        for epoch in range(self.max_iter):

            # 每轮打乱数据（真正的 SGD 需要）
            if self.shuffle:
                X, y = self._shuffle_data(X, y)

            # 随机梯度下降：逐样本更新
            for i in range(n_samples):
                xi = X[i]
                yi = y[i]

                # 预测
                y_pred = np.dot(xi, self.w) + self.b

                # MSE loss 的梯度：
                # d/dw = (y_pred - y) * x
                # d/db = (y_pred - y)
                error = y_pred - yi
                grad_w = error * xi
                grad_b = error

                # 参数更新（梯度下降）
                self.w -= self.lr * grad_w
                self.b -= self.lr * grad_b

        return self

    def predict(self, X):
        return np.dot(X, self.w) + self.b


In [2]:
# 生成数据
np.random.seed(0)
X = 2 * np.random.rand(100, 1)
y = 4 + 3 * X[:, 0] + np.random.randn(100)

# 训练 SGD
sgd = MySGDRegressor(lr=0.01, max_iter=20)
sgd.fit(X, y)

print("w:", sgd.w)
print("b:", sgd.b)


w: [2.9779956]
b: 4.172336317497534


In [1]:
import numpy as np

class SGD:
    """梯度法实现类，用于回归任务。
    """
    def __init__(self, eta=0.1, epochs=10):
        """
        Parameters
        ----------
        eta : float
            学习率。
        epochs : int
            训练的轮数。
        """
        self.eta = eta
        self.epochs = epochs
        
    def fit(self, X, y):
        """
        Parameters
        ----------
        X : array-like, shape=(n_samples, n_features)
            训练样本数据。
        y : array-like, shape=(n_samples, )
            样本标签。
        """
        X = np.asarray(X)
        y = np.asarray(y)
        # 随机初始化模型的参数（初始化点位置）。
        self.coef_ = np.random.random(size=X.shape[1])
        self.intercept_ = np.random.random()
        # 根据PEP8规范，对于没有使用的变量，定义为_。
        for _ in range(self.epochs):
            # 生成[0, X.shape[0] - 1]的随机排列。
            order = np.random.permutation(X.shape[0])
            # 每次获取一个样本，执行随机梯度下降。
            for xi, yi in zip(X[order], y[order]):
                # 计算当前样本的预测值。
                y_pred = np.dot(xi, self.coef_) + self.intercept_
                # 根据SGD的公式，来更新模型参数。w = w + eta * (yi - y_hat_i) * xi
                self.coef_ += self.eta * (yi - y_pred) * xi
                self.intercept_ += self.eta * (yi - y_pred)
        return self
    
    def predict(self, X):
        return np.dot(X, self.coef_) + self.intercept_

In [2]:
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

X, y, w = make_regression(n_samples=1000, n_features=5, coef=True, noise=2, random_state=23)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=222)
sgd = SGD(eta=0.1, epochs=5)
sgd.fit(X_train, y_train)
print(w)
print(sgd.coef_)
print(sgd.intercept_)
y_pred = sgd.predict(X_test)
print(r2_score(y_test, y_pred))

[19.77536323 23.2738     92.90040014 10.92580382 95.65233312]
[19.863627   23.42009519 92.4946959  10.90604385 96.766414  ]
-0.3635609941165926
0.9996435802028718


In [3]:
class SGD:
    """梯度法改进版本，可以让梯度在迭代的过程中，提前结束。
    """
    def __init__(self, eta=0.1, epochs=10, tol=1e-4, n_iter_no_change=5):
        """
        Parameters
        ----------
        eta : float
            学习率。
        epochs : int
            训练的轮数。
        tol : float
            容忍度。在梯度法迭代过程中，如果损失函数的下降幅度小于tol，则停止迭代。
        n_iter_no_change : int
            连续没有改进的次数。在梯度法迭代过程中，如果连续n_iter_no_change没有改进，则停止迭代。
            改进条件：当前损失函数的值，相对于历史最好（小）的损失函数值，下降幅度超过tol。
        """
        self.eta = eta
        self.epochs = epochs
        self.tol = tol
        self.n_iter_no_change = n_iter_no_change
        
    def compute_loss(self, X, y):
        """定义损失函数，计算损失函数的值。
        
        Parameters
        ----------
        X : array-like, shape=(n_samples, n_features)
            训练样本数据。
        y : array-like, shape=(n_samples, )
            样本标签。
            
        Returns
        -------
        loss : float
            所有样本的损失的平均值（MSE）。
        """
        y_pred = np.dot(X, self.coef_) + self.intercept_
        return np.mean((y - y_pred) ** 2)
        
    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        # 随机初始化模型的参数（初始化点位置）。
        self.coef_ = np.random.random(size=X.shape[1])
        self.intercept_ = np.random.random()
        # 存储梯度法迭代过程中，历史最好的损失。
        best_loss = np.inf
        for _ in range(self.epochs):
            order = np.random.permutation(X.shape[0])
            for xi, yi in zip(X[order], y[order]):
                # 计算当前迭代的损失值。
                loss = self.compute_loss(X, y)
                # 如果当前的损失，能够比历史最好的损失，下降幅度超过tol，则更新历史最好的损失。
                if best_loss - loss > self.tol:
                    # 使用当前的损失，来刷新历史最好的损失。
                    best_loss = loss
                    # 因为损失函数下降符合要求，因此，将连续没有改进的次数清零。
                    no_improvement_count = 0
                else:
                    # 否则，损失函数下降不符合要求，将连续没有改进的次数增1。
                    no_improvement_count += 1
                # 如果连续没有改进的次数，达到我们设置的阈值（n_iter_no_change），我们就认为已经到达了
                # 极值点附近，因此，停止迭代。
                if no_improvement_count >= self.n_iter_no_change:
                    return self
                y_pred = np.dot(xi, self.coef_) + self.intercept_
                self.coef_ += self.eta * (yi - y_pred) * xi
                self.intercept_ += self.eta * (yi - y_pred)
        print("程序没有收敛。")
        return self
    
    def predict(self, X):
        return np.dot(X, self.coef_) + self.intercept_

In [5]:
class SGD:
    def __init__(self, eta=0.1, epochs=10, tol=1e-4, n_iter_no_change=5):
        self.eta = eta
        self.epochs = epochs
        self.tol = tol
        self.n_iter_no_change = n_iter_no_change
        
    def compute_loss(self, X, y):
        y_pred = np.dot(X, self.coef_) + self.intercept_
        return np.mean((y - y_pred) ** 2)
        
    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        
        self.coef_ = np.random.random(size=X.shape[1])
        self.intercept_ = np.random.random()

        best_loss = np.inf
        no_improvement_count = 0   # ← 必须初始化！！

        for _ in range(self.epochs):
            order = np.random.permutation(X.shape[0])
            for xi, yi in zip(X[order], y[order]):

                # 计算全数据的 loss
                loss = self.compute_loss(X, y)

                # 是否改进？
                if best_loss - loss > self.tol:
                    best_loss = loss
                    no_improvement_count = 0
                else:
                    no_improvement_count += 1

                # 提前停止
                if no_improvement_count >= self.n_iter_no_change:
                    return self

                # 参数更新
                y_pred = np.dot(xi, self.coef_) + self.intercept_
                self.coef_ += self.eta * (yi - y_pred) * xi
                self.intercept_ += self.eta * (yi - y_pred)

        print("程序没有收敛。")
        return self
    
    def predict(self, X):
        return np.dot(X, self.coef_) + self.intercept_


In [10]:
sgd = SGD(eta=0.1, epochs=1)
sgd.fit(X_train, y_train)
print(w)
print(sgd.coef_)
print(sgd.intercept_)
y_pred = sgd.predict(X_test)
print(r2_score(y_test, y_pred))

[19.77536323 23.2738     92.90040014 10.92580382 95.65233312]
[22.83446001 23.81528656 88.85571859 13.71207227 94.30559959]
0.6035396757848683
0.9975888853990281
